# Capstone: Hand-Build the Attention Heads

Capstone notebook for the [Transformers & Attention](https://josephausterweil.github.io/probintro/deep/transformers-attention/) chapter (Part VIII).

**The experiment.** Attention is simple enough to build from scratch — and rich enough that *hand-set* keys and queries can implement real algorithms, the same ones researchers find inside trained models. You will:

- **(1)** write single-head attention yourself (dot-product scores → softmax with sharpness `beta` → weighted mix of values);
- **(2a)** hand-set keys and queries so every token attends to the *previous* token — a **previous-token head**;
- **(2b)** hand-set a **mini induction pattern**: on `katsu Fri burger Sat katsu`, the final `katsu` finds the *earlier* `katsu` and retrieves what followed it (`Fri`) — history, repeated;
- visualize every weight matrix as a **heatmap** (a previous-token head is a bright sub-diagonal stripe you can *see*);
- *(stretch)* compose the two heads across two layers — the **induction circuit**.

**Anchors.** Vaswani et al. (2017), *Attention is all you need*, is the source of the attention-only architecture you are rebuilding one head of. Olsson et al. (2022), *In-context learning and induction heads*, is the interpretability result you are miniaturizing: they found heads in trained transformers that locate a previous occurrence of the current token and attend to the token that came right after it, copying it forward; these heads emerge abruptly early in training, and the authors argue — on evidence they themselves call circumstantial, gathered from several directions — that they drive much of in-context learning. Here you set the weights by hand, so you know *exactly* why the pattern appears.

## 0. Setup (complete — run me)

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt

print('JAX ready:', jax.__version__)


def show_weights(W, tokens, title=""):
    """Heatmap of an attention weight matrix (rows = queries, cols = keys)."""
    fig, ax = plt.subplots(figsize=(4.6, 4.2))
    im = ax.imshow(W, cmap='viridis', vmin=0.0, vmax=1.0)
    ax.set_xticks(range(len(tokens)), tokens)
    ax.set_yticks(range(len(tokens)), tokens)
    ax.set_xlabel('key (looked-up token)')
    ax.set_ylabel('query (asking token)')
    for i in range(len(tokens)):
        for j in range(len(tokens)):
            v = float(W[i, j])
            ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                    color='white' if v < 0.5 else 'black', fontsize=8)
    ax.set_title(title)
    fig.colorbar(im, ax=ax, shrink=0.85)
    plt.show()

## 1. The toy sequence (complete — run me)

Five tokens from a Journal-flavored pattern: `katsu Fri burger Sat katsu`. Each token's embedding is two one-hot pieces glued together:

- **content** — which of the 4 distinct words it is (columns 0–3);
- **position** — which of the 5 slots it sits in (columns 4–8).

The position block is the chapter's "add position information" patch made maximally literal — without it, attention is order-blind and a "look one step back" head would be impossible to express.

In [ ]:
tokens = ['katsu', 'Fri', 'burger', 'Sat', 'katsu']
T = len(tokens)
vocab = ['katsu', 'Fri', 'burger', 'Sat']

content = jnp.stack([jnp.eye(len(vocab))[vocab.index(t)] for t in tokens])  # (5, 4)
position = jnp.eye(T)                                                       # (5, 5)
E = jnp.concatenate([content, position], axis=1)                            # (5, 9)

print('embeddings E:', E.shape, ' (columns 0-3 = content, 4-8 = position)')
print("row for the final 'katsu':", E[4])

## 2. Single-head attention from scratch — TODO

The chapter's formula, for a whole stack of tokens at once. With one query, key, and value per token,

$$\text{weight}_{ij} = \frac{\exp(\beta\, \mathbf{q}_i \cdot \mathbf{k}_j)}{\sum_{j'} \exp(\beta\, \mathbf{q}_i \cdot \mathbf{k}_{j'})}, \qquad \text{output}_i = \sum_j \text{weight}_{ij}\, \mathbf{v}_j.$$

Every row of the weight matrix is one softmax; the whole score table is one matrix product.

In [ ]:
def attention(q_mat, k_mat, v_mat, beta=1.0):
    """Single-head attention.

    q_mat: (T, d_k) one query per token    k_mat: (T, d_k) one key per token
    v_mat: (T, d_v) one value per token    beta:  sharpness of the softmax
    Returns (weights, output): weights is (T, T), rows summing to 1;
    output is (T, d_v) — each row is that query's weighted mix of the values.
    """
    # TODO — the chapter's three moves:
    #   1. scores = beta * (q_mat @ k_mat.T)              # all dot products at once
    #   2. softmax each ROW (subtract the row max first — the log-sum-exp guard):
    #        w = jnp.exp(scores - scores.max(axis=1, keepdims=True))
    #        w = w / w.sum(axis=1, keepdims=True)
    #   3. output = w @ v_mat
    #   return w, output
    raise NotImplementedError

# Smoke test, once attention() works (every row of weights must sum to 1):
#
# w, out = attention(E, E, E, beta=1.0)
# print(w.sum(axis=1))          # -> [1. 1. 1. 1. 1.]
# show_weights(w, tokens, 'raw embeddings attending to themselves')

## 3. (a) The previous-token head — TODO

Hand-set three matrices so that **every token attends exactly one position back**. The trick runs entirely on the position block of the embedding:

- $W_K$ should expose each token's **position** and nothing else — so a key says "I am slot $j$."
- $W_Q$ should turn "I am slot $i$" into the question "**who is slot $i-1$?**" — a *shifted* identity on the position block.
- $W_V$ should hand over **content** only — what a looked-up token actually contributes.

Then the score $\mathbf{q}_i \cdot \mathbf{k}_j$ is $1$ exactly when $j = i - 1$ and $0$ otherwise, and a large `beta` turns that margin into a near-hard lookup.

In [ ]:
# TODO (a): build the three matrices, then run the head.
#
#   W_K : (5, 9) — zeros on the 4 content columns, the identity on the 5
#         position columns.  Hint: jnp.concatenate([jnp.zeros((T, 4)), jnp.eye(T)], axis=1)
#   W_Q : (5, 9) — like W_K, but with the SHIFTED identity jnp.eye(T, k=1) on the
#         position block: it sends e_pos(i) -> e_pos(i-1).  (Position 0 has no
#         predecessor — its query becomes the zero vector. Look at row 0 of your
#         heatmap and explain what a zero query makes the softmax do.)
#   W_V : (4, 9) — the content columns, i.e. jnp.concatenate([jnp.eye(4), jnp.zeros((4, T))], axis=1)
#
# Then:
#   Qm, Km, Vm = E @ W_Q.T, E @ W_K.T, E @ W_V.T
#   W_prev, out_prev = attention(Qm, Km, Vm, beta=8.0)
#   show_weights(W_prev, tokens, 'previous-token head')
#
# Expected picture: a bright stripe one step BELOW the diagonal — every token
# looking at the token just before it. Check the payload too:
#   print(out_prev[4], '~', E[3, :4])   # the final katsu is handed "Sat"'s content

## 4. (b) A mini induction pattern — TODO

The induction move is: *when my own token has appeared before, find that earlier occurrence and retrieve what came right after it.* In trained transformers this takes **two heads in two layers** (Olsson et al., 2022): a previous-token head first writes "the token before me was X" into each position, and only then can an induction head match on it.

Here we do the first layer's job **by hand** (the stretch section does it for real): augment each token's embedding with its *predecessor's content* — exactly the information a previous-token head would have deposited there. One head then completes the pattern:

- $W_Q$: my **own** content — the final `katsu` asks "`katsu`?"
- $W_K$: my **predecessor's** content — `Fri`'s key says "the token before me was `katsu`."
- $W_V$: my **own** content — what gets copied forward.

So the final `katsu` (row 4) should attend to `Fri` (column 1) — the token that followed the *earlier* `katsu` — and the output should be (approximately) the `Fri` one-hot: *what happened last time, proposed again.*

In [ ]:
# Layer-1's work, done by hand: each row of pred_content is the CONTENT of the
# token one slot earlier (zeros for position 0, which has no predecessor).
pred_content = jnp.concatenate([jnp.zeros((1, len(vocab))), content[:-1]], axis=0)
E_aug = jnp.concatenate([content, position, pred_content], axis=1)   # (5, 13)
print('augmented embeddings:', E_aug.shape, ' (columns 9-12 = predecessor content)')

# TODO (b): three selector matrices over E_aug's columns (0-3 own content,
# 4-8 position, 9-12 predecessor content):
#   W_Q : select OWN content        (4 x 13)
#   W_K : select PREDECESSOR content (4 x 13)
#   W_V : select OWN content        (4 x 13)
# Then:
#   Qm, Km, Vm = E_aug @ W_Q.T, E_aug @ W_K.T, E_aug @ W_V.T
#   W_ind, out_ind = attention(Qm, Km, Vm, beta=8.0)
#   show_weights(W_ind, tokens, 'mini induction pattern')
#   print('prediction for the final katsu:', out_ind[4])   # ~ the Fri one-hot
#
# Reading the full heatmap: row 4 (katsu) lights on Fri — precedent found,
# history repeated. Rows whose token never occurred before have no matching key;
# what does the softmax do with an all-zero score row, and why is that honest?

## 5. Stretch: the two-layer induction circuit — TODO

No hand-augmentation this time. Compose the two heads you built, the way trained transformers do:

1. **Layer 1** = your previous-token head from part (a), run on the plain embeddings `E`. Its output hands each position its predecessor's content — *computed by attention, not by you*. Concatenate it onto the embeddings (a stand-in for the residual stream): `E1 = jnp.concatenate([E, out_prev], axis=1)`.
2. **Layer 2** = your induction head from part (b), reading queries from own-content columns and keys from the layer-1 columns of `E1`.

If layer 2's heatmap matches part (b)'s, you have reproduced — with every weight set by hand — the two-head induction circuit Olsson et al. (2022) describe inside trained models. **Questions to answer in a sentence each:** Why does the circuit *need* two layers (what information is missing from `E` alone)? Why does lowering `beta` in layer 1 degrade layer 2 more than lowering it in layer 2 does? And why would a *causal mask* (each row may only look left) leave this circuit intact?

In [ ]:
# TODO (stretch):
#   1. W_prev_1, out_prev = attention(E @ W_Q_a.T, E @ W_K_a.T, E @ W_V_a.T, beta=8.0)
#   2. E1 = jnp.concatenate([E, out_prev], axis=1)          # (5, 13): learned-by-attention
#   3. rerun part (b)'s selectors on E1 (columns 9-12 now come from layer 1)
#   4. show_weights for both layers, side by side — the circuit, end to end.


---

## References

- Olsson, C., Elhage, N., Nanda, N., Joseph, N., et al. (2022). In-context learning and induction heads. *Transformer Circuits Thread*. <https://transformer-circuits.pub/2022/in-context-learning-and-induction-heads/index.html>
- Vaswani, A., Shazeer, N., Parmar, N., Uszkoreit, J., Jones, L., Gomez, A. N., Kaiser, Ł., & Polosukhin, I. (2017). Attention is all you need. *NeurIPS*.

---

*This project is generously funded by the Japanese Probabilistic Computing Consortium Association ([JPCCA](https://jpcca.org/)).*